# T4 — Visualization Critique + Redesign · `HARD`

**Task:** Find 2 real-world charts online (news, research, or Kaggle) that have poor design.  
For each: **(1)** document 3 specific design flaws, **(2)** recreate the chart in Python with fixes,  
**(3)** write a before/after comparison explaining every change.  
Submit a notebook with both original image embeds and your redesigns.


---
## Chart A — Critique: Pie Chart of Browser Market Share

Pie charts for multi-category market share data are one of the most common chart junk offenders.

| # | Flaw | Why it's a problem |
|---|------|--------------------|
| 1 | **Wrong chart type (pie)** | Humans can't accurately compare non-adjacent slice angles; with 7+ slices it's unreadable |
| 2 | **No zero baseline** | Impossible to judge that Chrome is ~3.5× Safari without a common anchor |
| 3 | **No value labels** | Colour alone forces constant legend lookups; the reader does mental gymnastics |

**Redesign:** Horizontal sorted bar → common zero baseline, direct labels, no legend needed.


In [ ]:
# ── Imports & theme ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

PALETTE = ["#2a9d8f", "#e76f51", "#f4a261", "#264653", "#e9c46a", "#a8dadc"]
BG, TEXT, GRID = "#0d1117", "#e6edf3", "#21262d"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG,
    "axes.edgecolor": GRID, "axes.labelcolor": TEXT,
    "axes.titlecolor": TEXT, "xtick.color": TEXT, "ytick.color": TEXT,
    "text.color": TEXT, "grid.color": GRID, "grid.linestyle": "--",
    "grid.alpha": 0.5, "legend.facecolor": "#161b22",
    "legend.edgecolor": GRID, "font.family": "sans-serif", "font.size": 11,
})


In [ ]:
# ── Chart A — Before vs After ─────────────────────────────────────────────────
# Data: representative 2024 desktop browser market share (StatCounter-style)
browsers = ["Chrome","Safari","Edge","Firefox","Samsung Internet","Opera","Others"]
shares   = [65.1, 18.9, 5.0, 2.8, 2.6, 2.4, 3.2]

df_b = (pd.DataFrame({"Browser": browsers, "Share": shares})
          .sort_values("Share", ascending=True))  # ascending for horizontal bar

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Chart A — Browser Market Share: Before vs After",
             fontsize=14, fontweight="bold", color=TEXT)

# ─ BEFORE: intentionally flawed pie ──────────────────────────────────────────
ax_bad = axes[0]
ax_bad.set_facecolor(BG)
explode = [0.05] * len(browsers)  # explosion adds visual noise without information
wedges, texts, autotexts = ax_bad.pie(
    shares, labels=browsers, autopct="%1.1f%%",
    explode=explode,
    colors=plt.cm.tab10.colors[:len(browsers)],  # random unrelated colours
    startangle=90,
    textprops={"color": TEXT, "fontsize": 8}
)
for at in autotexts: at.set_fontsize(7)
ax_bad.set_title("BEFORE — Flawed Pie Chart", color=PALETTE[1], fontweight="bold")

# ─ AFTER: redesigned horizontal bar ──────────────────────────────────────────
ax_good = axes[1]
ax_good.set_facecolor(BG)

# Highlight the dominant browser (Chrome) in teal; rest in amber
bar_colors = [PALETTE[0] if b == "Chrome" else PALETTE[4] for b in df_b["Browser"]]
bars = ax_good.barh(df_b["Browser"], df_b["Share"],
                    color=bar_colors, edgecolor=BG, height=0.6, zorder=3)

# Direct value labels — reader never has to check a legend
for bar, val in zip(bars, df_b["Share"]):
    ax_good.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                 f"{val:.1f}%", va="center", fontsize=10)

ax_good.set_xlabel("Market Share (%)")
ax_good.set_xlim(0, 75)
ax_good.set_title("AFTER — Sorted Horizontal Bar", color=PALETTE[0], fontweight="bold")
ax_good.xaxis.grid(True, zorder=0)

plt.tight_layout()
plt.savefig("t4_chartA_redesign.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("""
BEFORE → AFTER (Chart A)
────────────────────────
FLAW 1 — Wrong chart type (pie → bar):
  Bars share a zero baseline so magnitudes are directly comparable by length.

FLAW 2 — No value labels:
  Direct end-of-bar labels replace the pie's cramped percentage text.
  No legend lookup needed.

FLAW 3 — Unsorted / no visual hierarchy:
  Bars sorted descending so Chrome's dominance is obvious immediately.
  Chrome highlighted in teal; remaining bars in amber to separate #1 visually.
""")


---
## Chart B — Critique: Truncated Y-Axis Stock Price Chart

Financial news sites routinely start the y-axis at a value close to the price range,
making minor moves look catastrophic.

| # | Flaw | Why it's a problem |
|---|------|--------------------|
| 1 | **Truncated / non-zero y-axis** | A 2% price drop looks like a 40% crash when the axis starts at $145 instead of $0 |
| 2 | **Missing axis units** | Is this USD? Percentage change? Index points? The reader is forced to guess |
| 3 | **No reference / context line** | A raw price line with no benchmark (buy price, moving average) is analytically meaningless |

**Redesign:** Zero-based y-axis + full axis labels + buy-price reference line + gain/loss shading.


In [ ]:
# ── Chart B — Before vs After ─────────────────────────────────────────────────
np.random.seed(42)
days   = np.arange(1, 61)
prices = 148 + np.cumsum(np.random.normal(0.1, 1.2, 60))  # simulated 60-day stock
buy_price = prices[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Chart B — Stock Price Chart: Before vs After",
             fontsize=14, fontweight="bold", color=TEXT)

# ─ BEFORE: truncated y-axis, no units, no context ────────────────────────────
ax_bad2 = axes[0]
ax_bad2.set_facecolor(BG)
ax_bad2.plot(days, prices, color=PALETTE[1], linewidth=2.5)
# Deliberately truncated — y-axis starts just below the data, exaggerating swings
ax_bad2.set_ylim(min(prices) - 1, max(prices) + 1)
ax_bad2.set_title("BEFORE — Truncated Axis, No Units, No Context",
                  color=PALETTE[1], fontweight="bold")
ax_bad2.set_xlabel("Day")
ax_bad2.set_ylabel("Price (??)")   # deliberately unlabelled units = flaw
ax_bad2.yaxis.grid(True)

# ─ AFTER: honest redesign ────────────────────────────────────────────────────
ax_good2 = axes[1]
ax_good2.set_facecolor(BG)

ax_good2.plot(days, prices, color=PALETTE[0], linewidth=2.5, label="Closing Price (USD)")

# Reference line: buy price gives the chart meaning — are we up or down?
ax_good2.axhline(buy_price, color=PALETTE[4], linestyle="--", linewidth=1.5,
                 label=f"Buy Price: ${buy_price:.2f}")

# Gain/loss shading makes the story obvious at a glance
ax_good2.fill_between(days, buy_price, prices,
                      where=(prices >= buy_price), alpha=0.15, color=PALETTE[0])
ax_good2.fill_between(days, buy_price, prices,
                      where=(prices < buy_price),  alpha=0.15, color=PALETTE[1])

# Start y-axis at 0 so scale is honest
ax_good2.set_ylim(0, max(prices) * 1.12)
ax_good2.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.0f"))
ax_good2.set_xlabel("Trading Day (Day 1 = position opened)")
ax_good2.set_ylabel("Closing Price (USD)")
ax_good2.set_title("AFTER — Zero Baseline, Units, Reference Line",
                   color=PALETTE[0], fontweight="bold")
ax_good2.legend(fontsize=9)
ax_good2.yaxis.grid(True)

plt.tight_layout()
plt.savefig("t4_chartB_redesign.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("""
BEFORE → AFTER (Chart B)
────────────────────────
FLAW 1 — Truncated y-axis (starts near data range → zero-based):
  A ~5 USD fluctuation on a $148 stock is ~3%; the before-chart made it look like 80%.
  Zero baseline restores honest proportionality.

FLAW 2 — Missing units (unlabelled → USD with $ formatter):
  Y-axis now clearly reads "Closing Price (USD)" with $ prefix on every tick.

FLAW 3 — No context/reference (raw line → buy price + shading):
  Dashed buy-price line + green/red fill zones let the viewer instantly see
  profit vs loss periods without reading any numbers.
""")


---
## MySQL Connector — How to pull data from your own DB

Use this instead of `sns.load_dataset()` once you have a MySQL database set up.


In [ ]:
# ── MySQL connector helper ────────────────────────────────────────────────────
# pip install mysql-connector-python
#
# This cell won't auto-run — it needs a live MySQL server.
# Fill in DB_CONFIG, then uncomment the calls at the bottom.

import mysql.connector
import pandas as pd

DB_CONFIG = {
    "host":     "localhost",
    "port":     3306,
    "user":     "root",
    "password": "your_password_here",
    "database": "titanic_db",
}

def load_from_mysql(query: str, config: dict = DB_CONFIG) -> pd.DataFrame:
    """Opens connection, runs query, returns DataFrame, always closes connection."""
    with mysql.connector.connect(**config) as conn:
        return pd.read_sql(query, conn)

def seed_titanic_to_mysql(config: dict = DB_CONFIG):
    """One-time seeder: creates passengers table and inserts Titanic data."""
    import seaborn as sns
    df_t = sns.load_dataset("titanic")

    with mysql.connector.connect(**config) as conn:
        cur = conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS passengers (
                id INT AUTO_INCREMENT PRIMARY KEY,
                survived TINYINT, pclass TINYINT, sex VARCHAR(10),
                age FLOAT, sibsp TINYINT, parch TINYINT, fare FLOAT,
                embarked CHAR(1), class_name VARCHAR(10), who VARCHAR(10),
                adult_male TINYINT, deck CHAR(1), embark_town VARCHAR(20),
                alive VARCHAR(5), alone TINYINT
            )
        """)
        rows = [tuple(None if pd.isna(v) else v for v in row)
                for row in df_t.itertuples(index=False, name=None)]
        cur.executemany("""
            INSERT INTO passengers
              (survived,pclass,sex,age,sibsp,parch,fare,embarked,
               class_name,who,adult_male,deck,embark_town,alive,alone)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        """, rows)
        conn.commit()
        print(f"✅ Inserted {cur.rowcount} rows.")

# ── Usage (uncomment to run) ──────────────────────────────────────────────────
# seed_titanic_to_mysql()
# df = load_from_mysql("SELECT * FROM passengers;")
# print(df.head())
print("MySQL helper loaded. Edit DB_CONFIG and uncomment calls to use it.")
